# Regular expression Note for 5G Cdu Elog Parse

- There are two CDU system capture DL and UL, which log will display different:
    - Split CDU using Ubuntu as system 
    - Flex CDU using CentOS as system 
- Case: There are different log file catpure differently, so I will mention some cases. 
    - Case1: parse Throughput of UL/DL throughput value, and RF signal information will look like this
        - SPlit: ` Tput=    0.000091 Mbps, Mcs=  4.0(Sigma= 0.0), Num=   1.4]`
        - Flex: ` Tput=   4.509057, Mcs=11.4, RB=  6.9, ReTx=  2.5, L=1.5, Bler=  3.1, A[8668]`   
- update:
    - 2024.09.01: Inital added project

- Description:
    - In this project, basically it's doing log file analysic, parse related text like DL for Downlink, or UL for Uplink 
    - I will be using alot of regular expression
    - in this page, showing some regular expression example to parse, for full code, please refer python file. 
    - I will get one line from log and make as string variable, in real code will read text file
- CDU
    - Have one system, but two diffrent OS and HW system, reason is split's performance for 256 Qam not reach, need to use split to be able to reach run higer performance for Downlink
- Libary: `import re`
    

## Case1 Parse CDU DL and UL Throughput (Layer2 Elog)

###  Split CDU parse related parameter

In [5]:
import re
stringSplit= "[20221018.165317.401606][info]:[DL- UE[ 0]: Tput=    0.000091 Mbps, Mcs=  4.0(Sigma= 0.0), Num=   1.4]"

- Step1: search for date and related string value

In [6]:
#get the date only
datesplit = stringSplit.split('[', 1)[1].split(']')[0] 
print(datesplit)

20221018.165317.401606


In [7]:
# search two result date and start from tput 
search = re.search(r'\[(\d+\.\d+\.\d+)\].*?(Tput=[^]]+)', stringSplit)
print(search.group(1)) #date
print(search.group(2)) # start from Tput until end

20221018.165317.401606
Tput=    0.000091 Mbps, Mcs=  4.0(Sigma= 0.0), Num=   1.4


**Note Pattern Explanation**

- get date `\[(\d+\.\d+\.\d+)\]`,  Matches `[1.2.3]`and capture 1.2.3, which is like our Date 
- Understand `.*?`:
    - `.*?` is used to match any characters between two specific parts of a pattern. In this example match the firt pattern and also the end pattern so will use this
    - `.`:match any character zero or more times.
    - `*`:match as few characters as possible while still allowing the overall pattern to match
- `(Tput=[^]]+)`:substring that starts with **Tput=** and continues until it end with character that is not **]**.

- Example: 

```python
pattern = re.compile(r"Start(.*?)End") 
text = "Start this is some text End“
match = pattern.search(text) 
if match: 
    print(match.group(1)) # Output: ' this is some text '
```


- Step2: remove the (Sigma= 0.0) , comma, leading space and split 

> search.group(2) : `Tput=    0.000091 Mbps, Mcs=  4.0(Sigma= 0.0), Num=   1.4`

In [8]:
#m3New= re.sub("[\[].*?[\]]", "",search.group(2)).replace(',','').strip().split()

# 2.1 remove (Sigma= 0.0)m search start with ( amd end with )
re.sub(r"[\(].*?[\)]", "", search.group(2)) #remove

# 2.2 remove comma, space, and split
# remove all comma 
#strip: Removes any leading or trailing whitespace from the string.
#split: Splits the string into a list of words based on whitespace.
search.group(2).replace(',','')
search.group(2).strip().split() 


# 2.3 add together 2.1 and 2.2 
m3New= re.sub(r"[\(].*?[\)]", "",search.group(2)).replace(',','').strip().split()
m3New
#['Tput=', '0.000091', 'Mbps', 'Mcs=', '4.0', 'Num=', '1.4']

['Tput=', '0.000091', 'Mbps', 'Mcs=', '4.0', 'Num=', '1.4']

- Step3: add to list and get the parameter

Will add element into new list, `getelement()` will put the key's value into res list. Like in my example, I have `Tput=`, `Mcs=` and so on. It will store the value of the key into res list. 

In [17]:
def getelement(li, element):
    ind = li.index(element)
    return li[ind+1]
res=[]
res.append(datesplit)
res.append(getelement(m3New, 'Tput='))
print(res)
# ['20221018.165317.401606', '0.000091']

['20221018.165317.401606', '282.747009']


###  Flex CDU parse related parameter

In [9]:
import re
stringFlex= "[20240829.133934.400015][info]:[D-UE[ 1][  1]: Tput=   4.095826, Mcs=11.3, RB=  6.8, ReTx=  3.3, L=1.4, Bler=  4.0, A[8732]"

- Step1: search for date and related string value

In [10]:
#get the date only
dateflex = stringSplit.split('[', 1)[1].split(']')[0] 
print(dateflex)

20221018.165317.401606


In [15]:
# search two result date and start from tput 
#search = re.search(r'\[(\d+\.\d+\.\d+)\].*?(Tput=[^]]+)', stringFlex)
search = re.search(r'\[(\d+\.\d+\.\d+)\].*?(Tput=[^A]+)', data)
print(search.group(1)) #date
print(search.group(2)) # start from Tput until end


20240605.182202.500824
Tput= 282.747009, Mcs=22.3, RB=256.9, ReTx=  0.0, L=3.9, Bler=  0.0, 


**Note Pattern Explanation**
 > - get date `\[(\d+\.\d+\.\d+)\]`,  Matches `[1.2.3]`and capture 1.2.3, which is like our Date 
 > - Understand `.*?`:
    - `.*?` is used to match any characters between two specific parts of a pattern. In this example match the firt pattern and also the end pattern so will use this
    - `.`:match any character zero or more times.
    - `*`:match as few characters as possible while still allowing the overall pattern to match
 > - `(Tput=[^A]+)`:substring that starts with **Tput=** and continues until it end with character that is not **]**. Plese refer below comaparison :
    ``` python
    # using pattern, start with Tput till the end with ]: (Tput=[^]]+)
    Tput=   4.095826, Mcs=11.3, RB=  6.8, ReTx=  3.3, L=1.4, Bler=  4.0, A[8732
    #using pattern, start with Tput and end with [A: (Tput=[^A]+)
    Tput= 282.747009, Mcs=22.3, RB=256.9, ReTx=  0.0, L=3.9, Bler=  0.0, 
    ```
    So I think you can use either one, but I think the second one is better. The last infomation we actually don't need it. 



- Step2: replace `=` with `= `space, and remove other character 

In order to let all the character be the same, need to set `=` to `= `, so I will use replace. After it I will remove comma, and remove any space. Finally split it


In [16]:
#search.group(2).replace('=', '= ').replace(',', ' ')
m3New = search.group(2).replace('=', r'= ').replace(',', ' ').strip().split()
print(m3New)

['Tput=', '282.747009', 'Mcs=', '22.3', 'RB=', '256.9', 'ReTx=', '0.0', 'L=', '3.9', 'Bler=', '0.0']


- Step3: add to list and get the parameter

Will add element into new list, `getelement()` will put the key's value into res list. Like in my example, I have `Tput=`, `Mcs=` and so on. It will store the value of the key into res list. 

In [18]:
def getelement(li, element):
    ind = li.index(element)
    return li[ind+1]
res=[]
res.append(datesplit)
res.append(getelement(m3New, 'Tput='))
print(res)
# ['20221018.165317.401606', '0.000091']

['20221018.165317.401606', '282.747009']


## Case2 other way to capture  parameter

I will use some other method to capture using different regualr expression module, I will be using base on Split 
CDU system 

you can go to regular expression to test match patterm: https://regex101.com/

In [23]:
cdudata = "[20221018.165317.401606][info]:[DL- UE[ 0]: Tput= 0.000091 Mbps, Mcs= 9.0(Sigma= 0.0), RbNum= 1.4, ReTxRatio= 33.3, Layers= 1.0, PdschBler= 0.0, nonWPdschBler= 33.3]"

### Method1 using re.search

> - **Purpose**: scans through a string, looking for the **first location** where the regular expression pattern produces a **match**.
> - **Return Value**: It returns a **match object if the pattern is found, or None if no match is found**. The match object contains information about the match, such as the **start and end positions and the matched text**.
> - **Use Case**: When you want to find the **first occurrence of a pattern and work** with it (e.g., extract or manipulate it).

- search matching pattern (the one we use in case1)

It will filter out the data in group(1), and Tput value in group(2)

In [25]:
m1 = re.search(r'\[(\d+\.\d+\.\d+)\].*?(Tput=[^]]+)', cdudata)
print(m1.group(2)) 
#group(1) => 20221018.165317.401606
#group(2) => Value=    0.000091 Mbps, Mcs=  4.0(Sigma= 0.0), Num=   1.4

Tput= 0.000091 Mbps, Mcs= 9.0(Sigma= 0.0), RbNum= 1.4, ReTxRatio= 33.3, Layers= 1.0, PdschBler= 0.0, nonWPdschBler= 33.3


- use split and get index

In this method it will split any white space, and use the index to get the value. This is **not a good method**, because if the **index or position of string chnage**, you have to **rewrite code**. 

In [28]:
#newstr=m1.group(2).split(' ')
#print(newstr)
print(m1.group(1),'\nTPUT:', newstr[1], '\nMcs',newstr[4].split('(')[0],'\nRb',newstr[7].split(',')[0])

20221018.165317.401606 
TPUT: 0.000091 
Mcs 9.0 
Rb 1.4


- getting matching group: add pattern of each paramter


In [101]:
search = re.search(r'\[(\d+\.\d+\.\d+)\].*?(Tput=[^]]+)', cdudata)
m4=re.search(r"^\[([\d\.]+).+Tput= ([\d\.]+).+Mcs= ([\d\.]+).+RbNum= ([\d.]+).+ReTxRatio= ([\d\.]+).+.", string)

# show all group result
print(m4.groups()) #('20221018.165317.401606', '0.000091', '9.0', '1.4', '33.3')

#show each group
print(m4.group(1)) # time 
print(m4.group(2)) # Tput
print(m4.group(3)) #Mcs
print(m4.group(4)) #RbNum
print(m4.group(5)) #ReTxRatio


('20221018.165317.401606', '0.000091', '9.0', '1.4', '33.3')
20221018.165317.401606
0.000091
9.0
1.4
33.3


In [102]:
#using loop display all result
if m4:
    for index, group in enumerate(m4.groups(), start=1):
        print(f"Group {index}: {group}")

Group 1: 20221018.165317.401606
Group 2: 0.000091
Group 3: 9.0
Group 4: 1.4
Group 5: 33.3


### Method2 using re.findall

> - **Purpose**: finds **all occurrences of a pattern** in a string and **returns them as a list of strings** (or tuples if the pattern has capturing groups).
> - **Return Value**: It **returns a list of all non-overlapping matches** of the pattern in the string. If **no matches** are found, it returns an **mpty list**.
> - **Use Case**: When you want to **find all instances of a pattern in a string** and work with all of them.

In [34]:
m2=re.findall(r"[0-9]*\.[0-9]+", cdudata) #date only first two field match, date, mintute,but second not match
print(m2)
newm2str=m2[2:-1] #get index from 2 to -1, the last second one.
print(newm2str)


['20221018.165317', '.401606', '0.000091', '9.0', '0.0', '1.4', '33.3', '1.0', '0.0', '33.3']
['0.000091', '9.0', '0.0', '1.4', '33.3', '1.0', '0.0']


In [108]:
#convert list to string
import re
print(newm2str)
m2tostr=" ".join(newm2str)
print(m2tostr)

['0.000091', '9.0', '0.0', '1.4', '33.3', '1.0', '0.0']
0.000091 9.0 0.0 1.4 33.3 1.0 0.0


### checking DL or UL 
in spit or flex in elog the UE's string name is name different:
- spit: UL-UE or DL-UE
- Flex: U-UE or D-UE

This is able to filter UL or DL data, or get specfic UL ID, so basically the ID inside UL or DL is the UE's ID. Each time the UE register will obtain a ID, so if UE drop and register again, it will gain a new ID. So you can fiter data using DL, UL, both DL and UL or with the ID

accepted_strings = re.compile(r"([DU]L\-\ UE(\[\s*(\d{1,2})\])?)|both$")
givenString = input("Please enter your search (Ex: DL- UE / UL- UE / UL- UE[ 0] / both:):")

In [141]:
import re
#accepted_strings = re.compile(r"([D]L\-\ UE(\[\ (\d)\])?)|both")
accepted_strings = re.compile(r"([DU]L\-\ UE(\[\s*(\d{1,2})\])?)|both$")
givenString = input("Please enter your search (Ex: DL- UE / UL- UE / UL- UE[ 0] / both:):")

#if accepted_strings.match(givenString):
if accepted_strings.match(givenString):
    if givenString =="both":
        UL = 'UL- UE'
        DL = 'DL- UE'
    else:
        print("you enter: ", givenString)
else:
    print("not found")

Please enter your search (Ex: DL- UE / UL- UE / UL- UE[ 0] / both:):UL- UE[ 1]
you enter:  UL- UE[ 1]


### Remove string re.sub

In [117]:
cdudata = "[20221018.165317.401606][info]:[DL- UE[ 0]: Tput= 0.000091 Mbps, Mcs= 9.0(Sigma= 0.0), RbNum= 1.4, ReTxRatio= 33.3, Layers= 1.0, PdschBler= 0.0, nonWPdschBler= 33.3]"

- Start from MCS= end at non: `re.search(r'\[(\d+\.\d+\.\d+)\].*?(Mcs=+[^n]+)', cdudata)`
> - `Mcs= 9.0(Sigma= 0.0), RbNum= 1.4, ReTxRatio= 33.3, Layers= 1.0, PdschBler= 0.0,`

In [121]:
res =re.search(r'\[(\d+\.\d+\.\d+)\].*?(Mcs=+[^n]+)', cdudata)
print(res.groups())
print(res.group(2))

('20221018.165317.401606', 'Mcs= 9.0(Sigma= 0.0), RbNum= 1.4, ReTxRatio= 33.3, Layers= 1.0, PdschBler= 0.0, ')
Mcs= 9.0(Sigma= 0.0), RbNum= 1.4, ReTxRatio= 33.3, Layers= 1.0, PdschBler= 0.0, 


- Remove `()`: Using replace to remove () or []

In [128]:
print(m3.group(2))
print(re.sub("[\(\[].*?[\)\]]", "",m3.group(2)))

Mcs= 9.0(Sigma= 0.0), RbNum= 1.4, ReTxRatio= 33.3, Layers= 1.0, PdschBler= 0.0, 
Mcs= 9.0, RbNum= 1.4, ReTxRatio= 33.3, Layers= 1.0, PdschBler= 0.0, 


- Remove using `\s` parameter

In [136]:
flexsampledata = "[20240605.182202.500824][info]:[D-UE[ 0][  0]: Tput= 282.747009, Mcs=22.3, RB=256.9, ReTx=  0.0, L=3.9"
search = re.search(r'\[(\d+\.\d+\.\d+)\].*?(Tput=[^]]+)', data)
key_value_str = search.group(2)
print(key_value_str)

# Remove spaces after '='
key_value_str_cleaned = re.sub(r'=\s+', '=', key_value_str)  #move spaces after '='
print(key_value_str_cleaned)

# Split by commas to create a list
s = key_value_str_cleaned.split(',')

result = []
for item in s:
    if '=' in item:
        key, value = item.split('=', 1)  # Split on the first '='
        result.append(key + '=')         # Keep the key with '='
        result.append(value)             # Append the value
    else:
        result.append(item)              # Append items without '='

print(result)

Tput= 282.747009, Mcs=22.3, RB=256.9, ReTx=  0.0, L=3.9, Bler=  0.0, A[6198
Tput=282.747009, Mcs=22.3, RB=256.9, ReTx=0.0, L=3.9, Bler=0.0, A[6198
['Tput=', '282.747009', ' Mcs=', '22.3', ' RB=', '256.9', ' ReTx=', '0.0', ' L=', '3.9', ' Bler=', '0.0', ' A[6198']
